# SmartCito Inference Demo

This notebook demonstrates how to run **SmartCito Model** inference with a user-supplied compatible base model plus SmartCito adapters.

This bundle does not include LLaMA-3 weights. It only ships SmartCito code, LoRA or QLoRA adapters, and synthetic or sovereign datasets. Users must obtain any compatible base model from official provider sources.

In [ ]:
from pathlib import Path
import json
import os

ROOT = Path.cwd()
BASE_MODEL_ID = os.getenv('SMARTCITO_BASE_MODEL_ID', '').strip()
ADAPTER_PATH = os.getenv('SMARTCITO_LORA_ADAPTER_PATH', str(ROOT / 'ai' / 'output' / 'smartcito-lora'))
API_BASE_URL = os.getenv('SMARTCITO_API_BASE_URL', 'http://localhost:8012')
BACKEND = os.getenv('SMARTCITO_LLM_BACKEND', 'merged-local').strip()

TASKS = [
    {
        'task': 'drone_mission_planning',
        'prompt': 'Plan a safe drone patrol route for a 2km radius around the stadium. Map: zone A/B/C; No-fly zones: north sector; Weather: clear; Wind: 8km/h.',
    },
    {
        'task': 'robot_navigation',
        'prompt': 'Summarize a robot navigation issue with low LiDAR confidence, elevated wheel slip, and a blocked south corridor.',
    },
    {
        'task': 'city_camera_analytics',
        'prompt': 'Interpret a city camera anomaly where two perimeter cameras detect motion and heat at North Gate with no badge scan.',
    },
    {
        'task': 'sensor_fusion',
        'prompt': 'Fuse river district sensor alerts with rising water level, heavy rainfall, degraded pumps, and increasing congestion.',
    },
    {
        'task': 'threat_reasoning',
        'prompt': 'Assess a high-confidence perimeter intrusion at a municipal power substation and recommend the immediate response.',
    },
]

print(json.dumps({
    'base_model_id': BASE_MODEL_ID or 'set SMARTCITO_BASE_MODEL_ID',
    'adapter_path': ADAPTER_PATH,
    'api_base_url': API_BASE_URL,
    'backend': BACKEND,
    'tasks': [item['task'] for item in TASKS],
}, indent=2))

## Local Inference

Use this path when the notebook environment has the base model and adapter available locally.

In [ ]:
import asyncio

async def run_local_inference():
    from ai.ai_models.llama_stack import generate_text

    if not BASE_MODEL_ID:
        raise RuntimeError('Set SMARTCITO_BASE_MODEL_ID before running local inference.')

    results = []
    for item in TASKS:
        payload = await generate_text(
            item['prompt'],
            system_prompt='You are SmartCito Model, a sovereign operations assistant for smart-city missions.',
            model=BASE_MODEL_ID,
            temperature=0.1,
            max_tokens=180,
            backend=BACKEND,
            adapter_path=ADAPTER_PATH,
            merge_lora=BACKEND == 'merged-local',
        )
        results.append({
            'task': item['task'],
            'provider': payload.get('provider'),
            'model': payload.get('model'),
            'response': payload.get('text'),
        })
    return results

# local_results = asyncio.run(run_local_inference())
# print(json.dumps(local_results, indent=2))

## API Inference

Use this path when the SmartCito FastAPI service is running and `/generate` is available.

In [ ]:
async def run_api_inference():
    import httpx

    results = []
    async with httpx.AsyncClient(timeout=120.0) as client:
        for item in TASKS:
            response = await client.post(
                f"{API_BASE_URL.rstrip('/')}/generate",
                json={
                    'prompt': item['prompt'],
                    'system_prompt': 'You are SmartCito Model, a sovereign operations assistant for smart-city missions.',
                    'model': BASE_MODEL_ID or None,
                    'backend': BACKEND,
                    'adapter_path': ADAPTER_PATH,
                    'merge_lora': BACKEND == 'merged-local',
                    'temperature': 0.1,
                    'max_tokens': 180,
                },
            )
            response.raise_for_status()
            payload = response.json()
            results.append({
                'task': item['task'],
                'provider': payload.get('provider'),
                'model': payload.get('model'),
                'response': payload.get('text'),
            })
    return results

# api_results = asyncio.run(run_api_inference())
# print(json.dumps(api_results, indent=2))

## Export Boundary

Share only adapter artifacts, evaluation reports, and notebooks. Do not upload foundation-model weights, secrets, or proprietary operational data.